NOTE: outputs excluded

# Import Libraries

In [ ]:
# Import necessary libraries
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from datetime import datetime
from collections import OrderedDict
from tqdm import tqdm

# Load Data

In [ ]:
import glob

base_path = r'DSSN2023/participants'

# Define column names for the DataFrame
column_names = ["Time", "Date", "ECG 1", "ECG 2", "Lateral Acc", "Longitud Acc", "Vertical Acc",
                "Chest Expansion", "Skin Temp", "GSR", "HR", "Inter-Beat Interval", "Video Tag"]

# Initialize an empty DataFrame to store all the data
combined_df = pd.DataFrame(columns=column_names + ["SourceFile"])

# Prepare a list for all participant folders
participant_folders = [os.path.join(base_path, f'{str(i).zfill(4)}/raw/equivital') for i in range(2, 40)]

# Loop through each participant's folder
for participant_folder in tqdm(participant_folders, desc='Processing participants'):
    print(participant_folder)
    # Check if the path exists to avoid errors
    if os.path.isdir(participant_folder):
        # Use glob to find all txt files in this directory
        files = glob.glob(os.path.join(participant_folder, '*.txt'))

        # Loop through the files and read them with a progress bar
        for file_path in tqdm(files, desc=f'Reading files in {os.path.basename(participant_folder)}'):
            # Extract the file name
            file_name = os.path.basename(file_path)
            print(file_name)
            # Read the text file into a DataFrame with the predefined column names
            file_df = pd.read_csv(file_path, sep='\t', skiprows=1, encoding="ISO-8859-1", header=None, names=column_names)

            # Add the file name as a new column
            file_df['SourceFile'] = file_name

            # Append to the main DataFrame
            combined_df = pd.concat([combined_df, file_df], ignore_index=True)

In [ ]:
combined_df['timestamp'] = combined_df['Date'] + ' ' + combined_df['Time']

# Now find the count of duplicate timestamps
duplicate_count = combined_df.duplicated(subset=['timestamp']).sum()
print("Count of duplicate timestamps:", duplicate_count)

# List of columns to be dropped
columns_to_drop = ["Time", "Date",
                   "Lateral Acc", "Longitud Acc", "Vertical Acc",
                   "Chest Expansion", "Skin Temp",
                   "HR", "Inter-Beat Interval", "Video Tag"]

# Drop the specified columns
combined_df.drop(columns_to_drop, axis=1, inplace=True)
combined_df[['participant', 'video #']] = combined_df['SourceFile'].str.extract(r'(\d{4})_vd(\d+)\.txt')

# Convert participant to integer
combined_df['participant'] = combined_df['participant'].astype(int)

# Convert video # to float
combined_df['video #'] = combined_df['video #'].astype(int)
combined_df['participant_video'] = combined_df['participant'].astype(str) + "_" + combined_df['video #'].astype(str)
combined_df

# Load Labels and Demographics

In [ ]:
excel_file_path = r'DSSN2023/participants/surveys_notes/post_experience_survey.xlsx'
survey_df = pd.read_excel(excel_file_path)

# Filter the DataFrame to include only rows where 'EXCL' column is NaN
survey_df = survey_df[survey_df['EXCL'].isna()]

# Convert participant to integer
survey_df['Participant #'] = survey_df['Participant #'].astype(int)

# Convert video # to float
survey_df['Video #'] = survey_df['Video #'].astype(int)

survey_df['participant_video'] = survey_df['Participant #'].astype(str) + "_" + survey_df['Video #'].astype(str)

# List of columns to be dropped
columns_to_drop = ["Duration (in seconds)", "StartDate", "EndDate",
                   "RecordedDate", "DO", "M", "EXCL"]

# Drop the specified columns
survey_df.drop(columns_to_drop, axis=1, inplace=True)
survey_df

In [ ]:
excel_file_path = r'DSSN2023/participants/surveys_notes/demographic_survey.xlsx'
demo_df = pd.read_excel(excel_file_path)

demo_df = demo_df.rename(columns={
    "What is your age?": "age",
    "What is your gender?": "sex",
    "What is the cultural and ethnic group you most identify with?": "ethnicity"
})

demo_small = demo_df[["Participant #", "age", "sex", "ethnicity"]]

survey_demo_df = survey_df.merge(demo_small, on="Participant #", how="left")
survey_demo_df

In [ ]:
merged_df = pd.merge(combined_df, survey_demo_df, on='participant_video', how='inner')

# List of columns to be dropped
columns_to_drop = ["SourceFile",  "video #", "Participant #", "Video #"] #"participant",

# Drop the specified columns
merged_df.drop(columns_to_drop, axis=1, inplace=True)
merged_df[['subject', 'video']] = merged_df['participant_video'].str.split('_', expand=True)
columns_to_drop = ['participant_video', 'participant']
merged_df = merged_df.drop(columns=columns_to_drop)
merged_df

# Exclude Participants

In [ ]:
# Exclude IDs 3, 4, 6, 7 as those subjects did not watch Videos 5 and 6.
df = merged_df.copy()
excluded_ids = [3, 4, 6, 7]
df = df[~df['subject'].isin(excluded_ids)]
df.reset_index(drop=True, inplace=True)
df

# Label encode and binarise labels

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit and transform the data
df['participant_trial_encoded'] = label_encoder.fit_transform(df['ID_video'])
df

In [ ]:
# Rename columns
df = df.rename(columns={'subject': 'ID'})
#df = df.rename(columns={'video': 'Trial'})

df['AR'] = pd.to_numeric(df['AR'], errors='coerce')
df['AR_Rating'] = df['AR'].apply(lambda x: 1 if x >= 5 else 0)
df['AR_Rating'] = pd.to_numeric(df['AR_Rating'])

df['VA'] = pd.to_numeric(df['VA'], errors='coerce')
df['VA_Rating'] = df['VA'].apply(lambda x: 1 if x >= 5 else 0)
df['VA_Rating'] = pd.to_numeric(df['VA_Rating'])
df = df.reset_index(drop=True)

df = df.sort_values(['ID', 'video'])
df['ID_video'] = df['ID'].astype(str) + '_' + df['video'].astype(str)
df

# Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

def scale_columns(group):
    scaler = StandardScaler()
    group[['ECG 1_scaled', 'ECG 2_scaled', 'GSR_scaled']] = scaler.fit_transform(group[['ECG 1', 'ECG 2', 'GSR']])
    return group

df = df.groupby('ID').apply(scale_columns).reset_index(drop=True)
df

# Final Export for modeling

In [ ]:
df.to_csv('DSSN_EQ_data_v3.csv')

# END